In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import optuna
import sys
from pathlib import Path
import torch

DIR_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(DIR_ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # use GPU if available, otherwise CPU

%load_ext autoreload
%autoreload 2

/home/koyo-ai/miniconda3/envs/homeless/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 987
TRACK = "off_market"                     # "off_market" or "list_aware"
VAL_START = pd.Timestamp("2025-12-01")    # sales on or after this date are validation
N_TRIALS = 10                            # hyperparameter optimization trials
N_ESTIMATORS = 2000                      # max boosting rounds per trial (just fancy way of saying epochs)
EARLY_STOPPING_ROUNDS = 100              # stop training if no improvement in validation loss after this many rounds

In [3]:
import data

listings = data.load("listings")
off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [4]:
import features

make_features = features.off_market_features if TRACK == "off_market" else features.list_aware_features

train = listings[listings["date_sold"] < VAL_START]
val = listings[listings["date_sold"] >= VAL_START]

X_train = make_features(data.hide_answers(train, TRACK))
X_val = make_features(data.hide_answers(val, TRACK))
y_train = np.log(train["sold_price"])
y_val = np.log(val["sold_price"])

print(f"train: {len(train):,} sales up to {train['date_sold'].max().date()}")
print(f"val:   {len(val):,} sales from {val['date_sold'].min().date()}")

train: 139,931 sales up to 2025-11-30
val:   10,069 sales from 2025-12-01


In [ ]:
def mdape(y_log, pred_log):
    return 100 * np.median(np.abs(np.exp(pred_log) / np.exp(y_log) - 1))

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model = XGBRegressor(**params, device=DEVICE, n_estimators=N_ESTIMATORS,
                         early_stopping_rounds=EARLY_STOPPING_ROUNDS, random_state=SEED)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    trial.set_user_attr("best_iteration", model.best_iteration)
    return mdape(y_val, model.predict(X_val))

In [6]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED), study_name="VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!")
study.optimize(objective, n_trials=N_TRIALS)

print("best val MdAPE:", round(study.best_value, 2))
print("best params:", study.best_params)
print("best iteration:", study.best_trial.user_attrs["best_iteration"])

[I 2026-10-15 19:54:50,678] A new study created in memory with name: VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!
[I 2026-10-15 19:55:39,423] Trial 0 finished with value: 12.391514305385565 and parameters: {'learning_rate': 0.03978007752343994, 'max_depth': 6, 'min_child_weight': 13.913328683467741, 'subsample': 0.7295646090144767, 'colsample_bytree': 0.9244579248120154, 'reg_lambda': 4.75284359207812}. Best is trial 0 with value: 12.391514305385565.
[I 2026-10-15 19:55:43,342] Trial 1 finished with value: 13.58042942358032 and parameters: {'learning_rate': 0.2225557983030707, 'max_depth': 10, 'min_child_weight': 1.0941794614139695, 'subsample': 0.5451291954754317, 'colsample_bytree': 0.9324386686870885, 'reg_lambda': 0.001846128974323298}. Best is trial 0 with value: 12.391514305385565.
[I 2026-10-15 19:55:59,287] Trial 2 finished with value: 12.061216903299998 and parameters: {'learning_rate': 0.14953417681650738, 'max_depth': 8, 'min_child_weight': 2.5163392983461716, 'subsample': 

KeyboardInterrupt: 

In [ ]:
model = XGBRegressor(**study.best_params, n_estimators=study.best_trial.user_attrs["best_iteration"] + 1,
                     random_state=SEED)
model.fit(X_train, y_train)

predicted = np.exp(model.predict(X_val))
error = 100 * np.abs(predicted / val["sold_price"] - 1)
print(f"{TRACK}  MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%")